# Pipeline orchestration: bronze -> silver -> gold

Runs the notebooks in order and stops at the first failure:

1. `03_stage1_drop_columns.ipynb` (bronze -> silver)
2. `04_stage2_clean_demo_subset.ipynb` (silver -> gold)

The source notebooks are left untouched. Each run writes an executed copy (with outputs) to `executed/`, so a failed run can be inspected cell by cell.

Run this notebook from the `notebooks` folder (the stage notebooks use relative paths).

In [ ]:
import time
from pathlib import Path

import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

NOTEBOOK_DIR = Path.cwd()
EXECUTED_DIR = NOTEBOOK_DIR / "executed"
EXECUTED_DIR.mkdir(exist_ok=True)

KERNEL = "hackatonel61"
CELL_TIMEOUT_S = 900

STEPS = [
    {
        "notebook": "03_stage1_drop_columns.ipynb",
        "expects": ["../data/silver/data_stage1.parquet"],
    },
    {
        "notebook": "04_stage2_clean_demo_subset.ipynb",
        "expects": [
            "../data/gold/apopo_demo_subset.parquet",
            "../data/gold/apopo_demo_subset.csv",
            "../data/gold/apopo_demo_data_dictionary.csv",
            "../data/gold/apopo_demo_cleaning_log.csv",
        ],
    },
]

assert (NOTEBOOK_DIR / "03_stage1_drop_columns.ipynb").exists(), "run this notebook from the notebooks folder"
assert (NOTEBOOK_DIR / "../data/bronze/data_bronze.xlsb").exists(), "bronze file missing: data/bronze/data_bronze.xlsb"

In [ ]:
def run_notebook(name):
    nb = nbformat.read(NOTEBOOK_DIR / name, as_version=4)
    client = NotebookClient(nb, kernel_name=KERNEL, timeout=CELL_TIMEOUT_S,
                            resources={"metadata": {"path": str(NOTEBOOK_DIR)}})
    try:
        client.execute()
        return None
    except CellExecutionError as exc:
        return exc
    finally:
        nbformat.write(nb, EXECUTED_DIR / name)


results = []
for step in STEPS:
    name = step["notebook"]
    print(f"running {name} ...", flush=True)
    start = time.time()
    error = run_notebook(name)
    seconds = round(time.time() - start, 1)

    if error is not None:
        results.append((name, "FAILED", seconds))
        print(f"FAILED after {seconds}s. Executed copy with the error: executed/{name}")
        print(str(error)[-1500:])
        break

    missing = [p for p in step["expects"] if not (NOTEBOOK_DIR / p).exists()]
    if missing:
        results.append((name, "MISSING OUTPUT", seconds))
        print("finished but expected output is missing:", missing)
        break

    results.append((name, "ok", seconds))
    print(f"done in {seconds}s")

In [ ]:
import pandas as pd

summary = pd.DataFrame(results, columns=["notebook", "status", "seconds"])
display(summary)

if len(summary) != len(STEPS) or (summary["status"] != "ok").any():
    raise RuntimeError("pipeline did not complete, see the output above")
print("pipeline completed")

## Output check

In [ ]:
silver = pd.read_parquet(NOTEBOOK_DIR / "../data/silver/data_stage1.parquet")
gold = pd.read_parquet(NOTEBOOK_DIR / "../data/gold/apopo_demo_subset.parquet")
cleaning_log = pd.read_csv(NOTEBOOK_DIR / "../data/gold/apopo_demo_cleaning_log.csv")

print(f"silver: {silver.shape[0]:,} rows x {silver.shape[1]} columns")
print(f"gold:   {gold.shape[0]:,} rows x {gold.shape[1]} columns, {gold['rat'].nunique()} rats")
cleaning_log